# Tutorial 3: Train NicheTrans on SMA data

In [ ]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_SMA import SMA

from utils.utils import *
from utils.utils_training_SMA import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [ ]:
%run ./args/args_SMA.py
args = args

set_seed(args.seed)
if torch.cuda.is_available():
    os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device: {}".format(device))
print("==========\nArgs:{}\n==========".format(args))

Using device: cpu
Args:Namespace(noise_rate=0.2, dropout_rate=0.1, n_source=3000, n_target=50, img_size=256, workers=4, path_img='/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used/patches', rna_path='/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used', msi_path='/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used', max_epoch=40, stepsize=20, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [ ]:
# create the dataloaders
dataset = SMA(path_img=args.path_img, rna_path=args.rna_path, msi_path=args.msi_path, n_top_genes=args.n_source, n_top_targets=args.n_target)
trainloader, testloader = sma_dataloader(args, dataset)

# create the model
# n_spot_types is determined by Leiden clustering in data_manager_SMA and must be
# passed here so the model allocates the correct number of per-type center tokens.
source_dimension, target_dimension = dataset.rna_length, dataset.msi_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension,
                   noise_rate=args.noise_rate, dropout_rate=args.dropout_rate,
                   n_spot_types=dataset.n_spot_types)
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

  Spot-type clustering: 10 types  (sizes: [875, 680, 396, 305, 259, 220, 169, 125, 53, 38])


  Spot-type clustering: 8 types  (sizes: [885, 706, 303, 251, 224, 210, 201, 138])


  Spot-type clustering: 8 types  (sizes: [882, 732, 255, 248, 189, 130, 120, 119])
  Slice 1 alignment: 8/8 clusters matched to reference
    local 0 -> ref 1 (global 1), cosine sim = 0.9972
    local 1 -> ref 8 (global 8), cosine sim = 0.9400
    local 2 -> ref 0 (global 0), cosine sim = 0.9000
    local 3 -> ref 5 (global 5), cosine sim = 0.9880
    local 4 -> ref 2 (global 2), cosine sim = 0.9840
    local 5 -> ref 3 (global 3), cosine sim = 0.9796
    local 6 -> ref 6 (global 6), cosine sim = 0.9851
    local 7 -> ref 4 (global 4), cosine sim = 0.9881
  Slice 2 alignment: 8/8 clusters matched to reference
    local 0 -> ref 0 (global 0), cosine sim = 0.9980
    local 1 -> ref 1 (global 1), cosine sim = 0.9979
    local 2 -> ref 4 (global 4), cosine sim = 0.9950
    local 3 -> ref 2 (global 2), cosine sim = 0.9965
    local 4 -> ref 6 (global 6), cosine sim = 0.9894
    local 5 -> ref 7 (global 7), cosine sim = 0.9166
    local 6 -> ref 9 (global 9), cosine sim = 0.9438
    local 7 

------Calculating spatial graph...


The graph contains 12134 edges, 3120 cells.
3.8891 neighbors per cell on average.


...

------Calculating spatial graph...


The graph contains 10360 edges, 2675 cells.
3.8729 neighbors per cell on average.


------Calculating spatial graph...


The graph contains 20628 edges, 2675 cells.
7.7114 neighbors per cell on average.


=> SMA loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  Without filtering  6038 spots from     2 slides 
  test     |  Without filtering  2675 spots from     1 slides
  train    |  After filting  6005 spots from     2 slides 
  test     |  After filting  2655 spots from     1 slides
  ------------------------------


### Initialize loss function (criterion) and optimizer

In [ ]:
criterion = nn.MSELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [ ]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, use_img=False, device=device)
    if args.stepsize > 0: scheduler.step()
    ################

pearson = test(model, testloader, use_img=False, device=device)
torch.save(model.state_dict(), 'NicheTrans_SMA_last.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/40


Batch 187/187	 Loss 60.682724 (90.212288)
==> Epoch 2/40


Batch 187/187	 Loss 31.780436 (41.400658)
==> Epoch 3/40


Batch 187/187	 Loss 35.736553 (17.299694)
==> Epoch 4/40


Batch 187/187	 Loss 6.646341 (13.186481)
==> Epoch 5/40


...

Batch 187/187	 Loss 3.609273 (4.427206)
==> Epoch 38/40


Batch 187/187	 Loss 3.879518 (4.444912)
==> Epoch 39/40


Batch 187/187	 Loss 3.851692 (4.410477)
==> Epoch 40/40


Batch 187/187	 Loss 5.024109 (4.431742)


Testing Set: pearson correlation 0.3890 + 0.1042; spearman correlation 0.4306 + 0.0651; rmse 2.2722 + 1.5582
Finished. Total elapsed time (h:m:s): 0:20:23
